# English → Chinese Translation
### Using the Google Cloud Translation API

**Before running, you need:**
1. A Google Cloud project with the **Cloud Translation API** enabled
2. An **API key** from APIs & Services → Credentials
3. Paste your key into Cell 2

> **Cost estimate:** Google charges $20 per 1M characters translated.
> For 10k short-to-medium emails, expect roughly $1–5 total.

## 1 — Install dependencies

In [ ]:
!pip install -q google-cloud-translate tqdm

## 2 — Set your API key

In [ ]:
# Paste your Google Cloud API key here
API_KEY = "YOUR_API_KEY_HERE"

if API_KEY == "YOUR_API_KEY_HERE":
    raise ValueError("Please replace YOUR_API_KEY_HERE with your actual API key.")

print("API key set.")

## 3 — Upload your JSON file

In [ ]:
from google.colab import files

print("Upload your chinese_sub_corpus.json file:")
uploaded = files.upload()

INPUT_FILENAME = list(uploaded.keys())[0]
print(f"\nUploaded: {INPUT_FILENAME}")

## 4 — Estimate cost before translating

In [ ]:
import json

# Load data — handles both JSON array and JSONL
with open(INPUT_FILENAME, "r", encoding="utf-8") as f:
    raw = f.read().strip()

try:
    data = json.loads(raw)
    if not isinstance(data, list):
        data = [data]
except json.JSONDecodeError:
    data = [json.loads(line) for line in raw.splitlines() if line.strip()]

indices = [i for i, item in enumerate(data) if item.get("text")]
texts   = [data[i]["text"] for i in indices]

total_chars = sum(len(t) for t in texts)
cost_usd    = (total_chars / 1_000_000) * 20

print(f"Records      : {len(texts):,}")
print(f"Total chars  : {total_chars:,}")
print(f"Estimated cost: ${cost_usd:.2f} USD")
print("\nProceed to Cell 5 to start translation.")

## 5 — Translate

> The Google Translate API accepts up to **128 strings per request**.
> Requests are retried automatically on transient errors.

In [ ]:
import os, time
import requests
from tqdm.notebook import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
BATCH_SIZE       = 128   # Google Translate API max per request
TARGET_LANG      = "zh"  # Simplified Chinese (use "zh-TW" for Traditional)
CHECKPOINT_FILE  = "translation_checkpoint.json"
CHECKPOINT_EVERY = 10    # save every N batches
MAX_RETRIES      = 3     # retry on API errors

TRANSLATE_URL = "https://translation.googleapis.com/language/translate/v2"

def translate_batch(batch: list[str]) -> list[str]:
    """Send a batch to the Google Translate API and return translations."""
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(
                TRANSLATE_URL,
                params={"key": API_KEY},
                json={
                    "q"      : batch,
                    "source" : "en",
                    "target" : TARGET_LANG,
                    "format" : "text",
                },
                timeout=30,
            )
            response.raise_for_status()
            translations = response.json()["data"]["translations"]
            return [t["translatedText"] for t in translations]

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait = 2 ** attempt   # exponential backoff: 1s, 2s, 4s
                print(f"Attempt {attempt + 1} failed ({e}). Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

# ── Resume from checkpoint if available ───────────────────────────────────────
results     = []
start_index = 0

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        results = json.load(f)
    start_index = len(results)
    print(f"Resuming from checkpoint: {start_index}/{len(texts)} already done.")

remaining = texts[start_index:]

# ── Translate ─────────────────────────────────────────────────────────────────
for batch_num, i in enumerate(tqdm(range(0, len(remaining), BATCH_SIZE), desc="Translating")):
    batch       = remaining[i : i + BATCH_SIZE]
    translated  = translate_batch(batch)
    results.extend(translated)

    # Periodic checkpoint
    if (batch_num + 1) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(results, f, ensure_ascii=False)

print(f"\nTranslation complete: {len(results)} records.")

## 6 — Save & download

In [ ]:
OUTPUT_FILENAME = "chinese_sub_corpus_translated.json"

# Write translations back into the original records
for idx, translation in zip(indices, results):
    data[idx]["text_zh"] = translation   # original "text" field is preserved

with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# Clean up checkpoint
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

# Download to your local machine
files.download(OUTPUT_FILENAME)
print(f"Downloaded: {OUTPUT_FILENAME}")